# DC-AC Converters

This chapter covers single-phase bridge inverters, sinusoidal PWM, and three-phase six-step operation in both `180^\circ` and `120^\circ` conduction modes.

## Theory Summary

### Single-phase square-wave inverters

A half-bridge produces `\pm V_{dc}/2`; a full-bridge produces `\pm V_{dc}`. The full-bridge square-wave output can be written as

$$v_o(\omega t)=\begin{cases}+V_{dc},&0<\omega t<\pi\\-V_{dc},&\pi<\omega t<2\pi\end{cases}$$

and its fundamental component has peak amplitude

$$V_{1,peak}=\frac{4V_{dc}}{\pi}.$$

### PWM ideas

For SPWM, a sinusoidal reference is compared against a high-frequency triangle carrier.

$$m_a=\frac{V_{ref,peak}}{V_{car,peak}}, \qquad m_f=\frac{f_c}{f_1}.$$

In the linear modulation region, the fundamental output is proportional to `m_a`.

- Bipolar SPWM switches the full bridge directly between `+V_{dc}` and `-V_{dc}`.
- Unipolar SPWM modulates the two legs separately, producing a three-level line voltage and pushing major switching harmonics upward.

### Three-phase six-step operation

In `180^\circ` conduction, each switch conducts for half a cycle and line voltages change every `60^\circ`.

In `120^\circ` conduction, one phase is open for `60^\circ` at a time, so only two phases conduct in each interval.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt

from power_sim import ac_dc, dc_ac, dc_dc, loads, plotter, utils

plt.rcParams["figure.figsize"] = (11, 8)
plt.rcParams["axes.grid"] = True

In [ ]:
f1 = 50.0
fs = 4500.0
t = utils.timebase(fundamental_frequency=f1, cycles=1.0, samples_per_cycle=4000)
omega_t = 2 * np.pi * f1 * t

square = dc_ac.single_phase_full_bridge_square(t, dc_bus=220.0, output_frequency=f1)
bipolar = dc_ac.single_phase_bipolar_spwm(t, dc_bus=220.0, output_frequency=f1, carrier_frequency=fs, modulation_index=0.85)
unipolar = dc_ac.single_phase_unipolar_spwm(t, dc_bus=220.0, output_frequency=f1, carrier_frequency=fs, modulation_index=0.85)

fig, axes = plotter.plot_stacked_waveforms(
    omega_t,
    [
        {"y": square["output_voltage"], "label": "Square-wave output", "ylabel": r"$v_o$ (V)", "color": "tab:blue"},
        {"y": bipolar["output_voltage"], "label": "Bipolar SPWM", "ylabel": r"$v_o$ (V)", "color": "tab:red"},
        {"y": unipolar["output_voltage"], "label": "Unipolar SPWM", "ylabel": r"$v_o$ (V)", "color": "tab:green"},
    ],
    xlabel=r"$\omega t$ (rad)",
    title="Single-phase inverter waveforms",
)
plt.show()

print("Bipolar m_f:", bipolar["m_f"])
print("Unipolar output RMS:", round(unipolar["metrics"]["rms"], 3), "V")

## Switching-State View

The three-phase inverter logic is intentionally separated from the load model. This lets you inspect pole voltages, phase voltages, and line voltages directly before adding any machine or passive-load dynamics.

In [ ]:
three_180 = dc_ac.three_phase_six_step(t, dc_bus=540.0, output_frequency=f1, conduction_mode=180)
three_120 = dc_ac.three_phase_six_step(t, dc_bus=540.0, output_frequency=f1, conduction_mode=120)

fig, axes = plotter.plot_stacked_waveforms(
    omega_t,
    [
        {"y": three_180["line_voltages"]["v_ab"], "label": r"$v_{ab}$, 180 deg", "ylabel": r"$v_{ab}$ (V)", "color": "tab:blue"},
        {"y": three_120["line_voltages"]["v_ab"], "label": r"$v_{ab}$, 120 deg", "ylabel": r"$v_{ab}$ (V)", "color": "tab:orange"},
        {"y": three_180["phase_voltages"][0], "label": r"$v_{an}$, 180 deg", "ylabel": r"$v_{an}$ (V)", "color": "tab:purple"},
    ],
    xlabel=r"$\omega t$ (rad)",
    title="Three-phase six-step inverter outputs",
)
plt.show()

print("180-degree line-voltage RMS:", round(three_180["metrics"]["rms"], 3), "V")
print("120-degree line-voltage RMS:", round(three_120["metrics"]["rms"], 3), "V")